# ai-detector retrain — ALL-IN-ONE (Stage-1 + Stage-2, v3 corpus)

One notebook, both stages. You upload 6 files (the corpus + the OOD gate + the two scripts), then **Run all**.

### Before running
- **Colab**: Runtime -> Change runtime type -> **T4 GPU**. The upload cell below pops a file picker.
- **Kaggle**: Settings -> Accelerator -> **GPU T4 x2**, and **Internet -> On**. Attach the 6 files as a Dataset (right panel -> + Add Input), and the upload cell will find them under /kaggle/input automatically.

### The 6 files to upload (paths on your Mac)
| upload as | path in the repo |
|---|---|
| `train.csv` | `data/corpus/train.csv` |
| `eval.csv` | `data/corpus/eval.csv` |
| `crossgen_eval.csv` | `data/corpus/crossgen_eval.csv` |
| `ood_human.csv` | `data/corpus/ood_human.csv` |
| `finetune-lora.py` | `scripts/finetune-lora.py` |
| `audit-confound.py` | `scripts/audit-confound.py` |

Ship signal per stage: **OOD false-positive rate drops** vs that stage's base with no register regressing, AND **cross-gen recall stays high**. The v3 corpus trains on 18-19c literary, political, scientific, and oratory prose (incl. Lincoln's speeches), so the OOD number should fall sharply from the 39% the v2 corpus produced.


## 0. Upload (Colab) or auto-find (Kaggle) the 6 files
Colab pops a picker. Kaggle finds them under /kaggle/input if you attached them as a Dataset. Re-run until it prints OK.


In [ ]:
import os, glob, shutil
need = ['train.csv','eval.csv','crossgen_eval.csv','ood_human.csv','eval_held.csv','finetune-lora.py','audit-confound.py']
# Kaggle: copy from an attached dataset
for f in [f for f in need if not os.path.exists(f)]:
    hits = glob.glob('/kaggle/input/**/'+f, recursive=True)
    if hits: shutil.copy(hits[0], f)
missing = [f for f in need if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Pick these', len(missing), 'files:', missing)
        files.upload()
    except Exception:
        print('Not on Colab and not found under /kaggle/input -> attach them as a Kaggle Dataset.')
missing = [f for f in need if not os.path.exists(f)]
print('\nOK - all 6 present' if not missing else 'STILL MISSING: ' + str(missing))
for f in need:
    if os.path.exists(f): print('  ', f, os.path.getsize(f), 'bytes')


## 1. Dependencies
Colab/Kaggle torch already has CUDA. Pin `transformers==4.49.0` (matches the Mac-side convert). torchao 0.10 ships preinstalled and breaks peft's LoRA path, so remove it.


In [ ]:
!pip -q install "transformers==4.49.0" "peft>=0.11" "accelerate>=0.30" scikit-learn
!pip uninstall -y torchao
import importlib.util, torch
print("torchao gone:", importlib.util.find_spec("torchao") is None, "| CUDA:", torch.cuda.is_available())


## 2. Calibration-view helper
Reports OOD false-positive rate at the *shipped* P(AI) thresholds (the gates below use argmax-0.5). If FP collapses at higher thresholds, calibration plus an abstain band rescues it.


In [ ]:
def pai_table(M):
    import torch, csv, numpy as np
    from collections import defaultdict
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    tok = AutoTokenizer.from_pretrained(M)
    model = AutoModelForSequenceClassification.from_pretrained(M).to(dev).eval()
    id2 = {int(k): v for k, v in model.config.id2label.items()}
    human_idx = next((i for i, n in id2.items() if 'human' in str(n).lower()), 0)
    rows = [r for r in csv.DictReader(open('ood_human.csv')) if r.get('text','').strip()]
    pAI, regs = [], []
    with torch.no_grad():
        for i in range(0, len(rows), 32):
            ch = rows[i:i+32]
            enc = tok([r['text'] for r in ch], truncation=True, max_length=512, padding=True, return_tensors='pt').to(dev)
            p = torch.softmax(model(**enc).logits, -1).cpu()
            for pr, row in zip(p, ch):
                pAI.append(1.0 - pr[human_idx].item()); regs.append(row.get('register','?'))
    pAI = np.array(pAI)
    print(f'[{M}] OOD humans={len(pAI)}  mean P(AI)={pAI.mean():.3f}')
    for tau in [0.5,0.7,0.85,0.9,0.93,0.95]:
        print(f'  P(AI)>{tau:.2f}: {int((pAI>tau).sum()):3d}/{len(pAI)} = {(pAI>tau).mean()*100:4.1f}%')
    agg = defaultdict(lambda:[0,0])
    for p,reg in zip(pAI,regs): agg[reg][1]+=1; agg[reg][0]+=int(p>0.90)
    print('  per-register FP at P(AI)>0.90:')
    for reg in sorted(agg, key=lambda k:-agg[k][0]/max(1,agg[k][1])):
        f,t = agg[reg]; print(f'    {reg[:18]:18s} {f}/{t} = {f/t*100:4.0f}%')


## 2b. Held-out eval helper
Per-register false-positive (fresh human) and recall (fresh AI) on `eval_held.csv` — works and articles the model never trained on, so this is the honest generalization number, especially for encyclopedic.


In [ ]:
def held_eval(M):
    import torch, csv
    from collections import defaultdict
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    dev='cuda' if torch.cuda.is_available() else 'cpu'
    tok=AutoTokenizer.from_pretrained(M)
    model=AutoModelForSequenceClassification.from_pretrained(M).to(dev).eval()
    id2={int(k):v for k,v in model.config.id2label.items()}
    hidx=next((i for i,n in id2.items() if 'human' in str(n).lower()),0)
    rows=[r for r in csv.DictReader(open('eval_held.csv')) if r.get('text','').strip()]
    P=[]
    with torch.no_grad():
        for i in range(0,len(rows),32):
            ch=rows[i:i+32]
            enc=tok([r['text'] for r in ch],truncation=True,max_length=512,padding=True,return_tensors='pt').to(dev)
            pr=torch.softmax(model(**enc).logits,-1).cpu()
            for p in pr: P.append(1.0-p[hidx].item())
    agg=defaultdict(lambda:[0,0,0,0])  # hfp,hn,rec,an
    for r,p in zip(rows,P):
        a=agg[r['register']]
        if r['label']=='0': a[1]+=1; a[0]+=int(p>0.90)
        else: a[3]+=1; a[2]+=int(p>0.90)
    print(f'[{M}] held-out eval (source-disjoint) @ P(AI)>0.90 — FP=human flagged, REC=AI caught')
    for reg in sorted(agg):
        hfp,hn,rec,an=agg[reg]
        print(f'  {reg:15s} FP {hfp:3d}/{hn:<3d} = {hfp/max(1,hn)*100:4.0f}%    REC {rec:3d}/{an:<3d} = {rec/max(1,an)*100:4.0f}%')


# Stage-1 — e5-small (33M, binary). Fast, calibratable, the register-bias source.


### 1a. Fine-tune (no --base => defaults to the e5-small detector)


In [ ]:
!python finetune-lora.py \
  --train train.csv --eval eval.csv \
  --seq 512 --batch 32 --epochs 3 --lr 2e-4 --save-steps 300 \
  --out out/stage1-ft


### 1b. OOD gate — base vs fine-tune (the Lincoln/Austen regression)


In [ ]:
!python audit-confound.py --ood ood_human.csv --model MayZhou/e5-small-lora-ai-generated-detector


In [ ]:
!python audit-confound.py --ood ood_human.csv --model out/stage1-ft/merged


### 1c. Calibration view (shipped P(AI) thresholds)


In [ ]:
pai_table('out/stage1-ft/merged')


### 1d. Cross-generator recall


In [ ]:
!python audit-confound.py --crossgen crossgen_eval.csv --model out/stage1-ft/merged


In [ ]:
held_eval('out/stage1-ft/merged')


### 1e. Download Stage-1 merged model


In [ ]:
import shutil
shutil.make_archive('stage1-ft', 'zip', 'out/stage1-ft/merged')
print('wrote stage1-ft.zip')
try:
    from google.colab import files; files.download('stage1-ft.zip')
except Exception:
    print('Kaggle: stage1-ft.zip is in the working dir -> download it from the Output tab.')


# Stage-2 — ModernBERT (Donnyed, 3-class head). Higher ceiling, slower, no calibration block.


### 2a. Fine-tune (`--batch 8` is T4-safe; raise on P100/A100)


In [ ]:
!python finetune-lora.py \
  --base Donnyed/LLM_Detector_Preview_model \
  --train train.csv --eval eval.csv \
  --seq 512 --batch 8 --epochs 2 --lr 2e-4 --save-steps 200 \
  --out out/stage2-ft


### 2b. OOD gate — base vs fine-tune


In [ ]:
!python audit-confound.py --ood ood_human.csv --model Donnyed/LLM_Detector_Preview_model


In [ ]:
!python audit-confound.py --ood ood_human.csv --model out/stage2-ft/merged


### 2c. Calibration view


In [ ]:
pai_table('out/stage2-ft/merged')


### 2d. Cross-generator recall


In [ ]:
!python audit-confound.py --crossgen crossgen_eval.csv --model out/stage2-ft/merged


In [ ]:
held_eval('out/stage2-ft/merged')


### 2e. Download Stage-2 merged model


In [ ]:
import shutil
shutil.make_archive('stage2-ft', 'zip', 'out/stage2-ft/merged')
print('wrote stage2-ft.zip')
try:
    from google.colab import files; files.download('stage2-ft.zip')
except Exception:
    print('Kaggle: stage2-ft.zip is in the working dir -> download it from the Output tab.')


# On your Mac (convert + install)
```bash
cd ~/Code/ai-detector
pip install "transformers==4.49.0" "coremltools>=9"

# Stage-1 (cleaner: true binary head + calibration step)
unzip ~/Downloads/stage1-ft.zip -d out/stage1-ft/
python3 scripts/convert-model.py out/stage1-ft/merged          # writes into Models/
python3 scripts/calibrate.py --model out/stage1-ft/merged --data calib.csv

# Stage-2 (3-class head -> binary P(AI)=1-P(human); check ai_label_index=1 after convert)
unzip ~/Downloads/stage2-ft.zip -d out/stage2-ft/
python3 scripts/convert-stage2-modernbert.py out/stage2-ft/merged
cp -r <printed converted .mlmodelc + jsons> Models/Stage2/

bash scripts/install.sh   # deploy + relaunch the installed app
```

For `calib.csv`, a small `text,label` slice of `eval.csv` plus a few `ood_human.csv` rows is a fine start.
